In [1]:
import pandas as pd
import numpy as np

# Download the matrices from:  https://drive.google.com/drive/folders/1-2og2FAM0_6e3L2_9C7HnOm9sgfrJIRe?usp=sharing
# Load the QUBO matrix and genes data (compute this on the MATLAB code)
df_genes = pd.read_csv('genes.csv', header=None)
df = pd.read_csv('qubo_matrix.csv', header=None)

# Convert QUBO matrix to numpy array
Qmat = np.asarray(df)
print("QUBO matrix shape:", Qmat.shape)

# Convert QUBO matrix to dictionary format
def qubo_matrix_to_dict(Q):
    qubo = {}
    for i in range(Q.shape[0]):
        for j in range(Q.shape[1]):
            if Q[i, j] != 0:
                qubo[(i, j)] = Q[i, j]
    return qubo

qubo = qubo_matrix_to_dict(Qmat)


QUBO matrix shape: (5000, 5000)


In [2]:
from dwave.system import LeapHybridSampler  # Import for the Leap Hybrid Sampler
import dimod  # Import for working with QUBOs

# Quantum (Hybrid) Solver
print("Quantum (Hybrid) Solver Results...")

# Using Leap Hybrid Sampler
sampler_q = LeapHybridSampler()
sampleset_q = sampler_q.sample_qubo(qubo)

# Retrieve and analyze quantum results
#print("Best solution: ", sampleset_q.first.sample)
print("Energy: ", sampleset_q.first.energy)
print("Occurrences: ", sampleset_q.first.num_occurrences)

# Convert total runtime from microseconds (10^-6) to seconds
runtime_all = sampleset_q.info.get('run_time', 0)  # Provide a default value in case the key is missing
runtime_all_seconds = runtime_all / 1_000_000

# Convert QPU access time from microseconds (10^-6) to seconds
annealing_time = sampleset_q.info.get('qpu_access_time', 0)  # Provide a default value in case the key is missing
annealing_time_seconds = annealing_time / 1_000_000

print(f"Quantum annealing time: {annealing_time_seconds} seconds")
print(f"D-Wave hybrid solver time: {runtime_all_seconds} seconds")


Quantum (Hybrid) Solver Results...
Energy:  -2.359038467439544
Occurrences:  1
Quantum annealing time: 0.079389 seconds
D-Wave hybrid solver time: 14.233201 seconds


In [3]:
# Store selected features from quantum solution
sample_dict_q = sampleset_q.first.sample
values_q = list(sample_dict_q.values())

# Convert list to numpy array and then cast to int
qa_sol = np.array(values_q).astype(int)

# Multiply selected rows by qa_sol (vector of ones, so it's effectively the sum of selected rows)
ener_per_feat = Qmat @ qa_sol

# Get dataframe with gene's information
df_genes['feature_selected'] = values_q
df_genes = df_genes.rename(columns={0: 'Gene'})  # Ensure '0' column is correctly named
df_genes['gene_score'] = ener_per_feat

# Filter results
filt_df_q = df_genes[df_genes['feature_selected'] > 0].copy()

# Sort by 'gene_score'
filt_df_q = filt_df_q.sort_values(by='gene_score', ascending=True).copy()

new_df = filt_df_q.reset_index(drop=True)
new_df.index = new_df.index.astype(str)
new_df.to_csv('filt_df_QA.csv', index=True)

In [2]:

# Import necessary libraries
from dwave.samplers import SimulatedAnnealingSampler
import dimod
import numpy as np
import pandas as pd

# Classical Solver with TabuSampler
print("\nClassical Solver Results:")

# Initialize the Tabu sampler
sampler_c = SimulatedAnnealingSampler()

# Sample the QUBO
sampleset_c = sampler_c.sample_qubo(qubo, num_reads=100)

# Retrieve and analyze classical results
print("Energy: ", sampleset_c.first.energy)
print("Occurrences: ", sampleset_c.first.num_occurrences)


Classical Solver Results:
Energy:  -2.359038466698621
Occurrences:  1


In [8]:
from dwave.samplers import SteepestDescentSolver
import dimod
import numpy as np
import pandas as pd

print("\nClassical Solver Results:")

# Initialize the Tabu sampler
sampler_c = SteepestDescentSolver()

# Sample the QUBO
sampleset_c = sampler_c.sample_qubo(qubo, num_reads=100)

# Retrieve and analyze classical results
print("Energy: ", sampleset_c.first.energy)
print("Occurrences: ", sampleset_c.first.num_occurrences)


Classical Solver Results:
Energy:  -2.359038466698621
Occurrences:  1


In [6]:
# Import necessary libraries
from dwave.samplers import TabuSampler
import dimod
import numpy as np
import pandas as pd

# Classical Solver with TabuSampler
print("\nClassical Solver Results:")

# Initialize the Tabu sampler
sampler_c = TabuSampler()

# Sample the QUBO
sampleset_c = sampler_c.sample_qubo(qubo, num_reads=10)

# Retrieve and analyze classical results
print("Energy: ", sampleset_c.first.energy)
print("Occurrences: ", sampleset_c.first.num_occurrences)


Classical Solver Results:
Energy:  8305.41150108588
Occurrences:  1


In [8]:
# Embedding problem for this large matrix, it is too complex for the available resources.
# import dimod
# import dwave.system
## Create a sampler using D-Wave's EmbeddingComposite
# sampler_q0 = dwave.system.EmbeddingComposite(dwave.system.DWaveSampler())
## Sample the QUBO
#sampleset_q0 = sampler_q0.sample_qubo(qubo, num_reads=100)
## Print sample results for QUBO (optional)
#print(sampleset_q0)